#### Imports

In [ ]:
# Imports
import os
import sys
from tqdm import tqdm
from ultralytics import YOLO

#### Path Configurations

In [ ]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Path Configuration
dataset_name = "premier_league_highlights_2026"
data_directory = os.path.join(project_root, "data")
model_directory = os.path.join(project_root, "models", "detection")

# Unannotated images directory
unannotated_images_data_directory = os.path.join(data_directory, "to_annotate", dataset_name, "images")
unannotated_labels_data_directory = os.path.join(data_directory, "to_annotate", dataset_name, "labels")
os.makedirs(unannotated_labels_data_directory, exist_ok=True)

#### Model Setup

In [ ]:
# Model Name and Weights
base_model_name = "23-03-2026_12-08_yolo26l"
full_model_weights_path = os.path.join(model_directory , base_model_name, "weights", "best.pt")
detection_model = YOLO(full_model_weights_path) # Can additionally load from a saved point
detection_model.eval()

#### Object Annotations Loop

In [ ]:
# Pre-collect files so tqdm knows the total count
jpg_files = [
    (root, file)
    for root, dirs, files in os.walk(unannotated_images_data_directory)
        for file in files
            if file.endswith(".jpg")
]

In [ ]:
# Creating Annotations
for root, file in tqdm(jpg_files, desc="Annotating images", unit="img"):
    # Gathering Correct Paths   
    image_file_name = file[:-4]
    full_image_path = os.path.join(unannotated_images_data_directory, f"{image_file_name}.jpg")
    
    # Label Directory handling
    full_label_path = os.path.join(unannotated_labels_data_directory, f"{image_file_name}.txt")

    # Ensure no training is performed
    detection_model.eval()
    detection_results = detection_model.predict(full_image_path, imgsz=1280, conf=0.25, verbose=False)

    # Preparing Annotations
    annotation_output = ''
    if detection_results[0].boxes:
        for index, class_id in enumerate(detection_results[0].boxes.cls):
            boxes_list = detection_results[0].boxes.xywhn[index]
            class_str = str(int(class_id.cpu().item()))
            bbox_list = boxes_list.cpu().tolist()
            x, y, w, h = bbox_list[0], bbox_list[1], bbox_list[2], bbox_list[3] 
            annotation_output += f"{str(class_str)} {round(x, 7)} {round(y, 7)} {round(w, 7)} {round(h, 7)}\n"
    with open(full_label_path, "w") as f:
        f.write(annotation_output)
